# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [ ]:
import os
from getpass import getpass
from langchain.vectorstores import ElasticsearchStore
from langchain_openai.embeddings import OpenAIEmbeddings

# 👉 Define tus claves directamente como strings (NO RECOMENDADO en producción)
ELASTIC_CLOUD_ID = ""
ELASTIC_API_KEY = ""
OPENAI_API_KEY = ""

# 👉 Establece la clave de OpenAI como variable de entorno
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# 👉 Instancia embeddings
embeddings = OpenAIEmbeddings()

# 👉 Instancia ElasticsearchStore
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="test",
    embedding=embeddings
)


C:\Users\Zahinos\AppData\Local\Temp\ipykernel_18804\3743859212.py:18: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use :class:`~Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [5]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=50 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [6]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="test",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [7]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
{page_content}
---
"""
)

def _combine_documents(docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)

# Crear el pipeline de QA
_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

# Lista de preguntas para lanzar al sistema
questions = [
    "What is the NASA sales team?",
    "How does the HR department support employee well-being in this company?",
    "What are the main responsibilities of the product design team according to the documents?"
]

# Ejecutar el pipeline con cada pregunta
for q in questions:
    print(f"\n--- Question: {q} ---\n")
    answer = chain.invoke(q)
    print(answer)
    print("\n" + "-"*60)



--- Question: What is the NASA sales team? ---



INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team operate within NASA?', '3. What are the responsibilities of the NASA sales team?']



The NASA sales team is a part of the Americas region in the sales organization. It is responsible for serving customers in North America and South America, and is led by two Area Vice-Presidents, Laura Martinez and Gary Johnson. The team works closely with other departments to identify and pursue new business opportunities, maintain existing client relationships, and ensure customer satisfaction.

------------------------------------------------------------

--- Question: How does the HR department support employee well-being in this company? ---



INFO:langchain.retrievers.multi_query:Generated queries: ['1. What measures does the HR department take to promote employee well-being within this organization?', '2. In what ways does the HR department provide support for employee well-being in this company?', '3. Can you explain the role of the HR department in ensuring employee well-being within this organization?']



The HR department supports employee well-being in this company by encouraging employees to prioritize their health and well-being while working from home. This includes taking regular breaks, maintaining a work-life balance, and seeking support from supervisors and colleagues when needed. The HR department also reviews and updates the work-from-home policy periodically, taking into account changes in public health guidance, business needs, and employee feedback. Additionally, employees are encouraged to direct any questions or concerns about this policy to their supervisor or the HR department.

------------------------------------------------------------

--- Question: What are the main responsibilities of the product design team according to the documents? ---



INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the key duties of the product design team as outlined in the documents?', '2. Can you provide a breakdown of the primary tasks assigned to the product design team based on the documents?', '3. According to the documents, what are the main obligations of the product design team?']



The main responsibilities of the product design team, as outlined in the documents, include understanding customer needs and pain points, explaining technical aspects of the product in simple language, addressing customer concerns and questions, seeking feedback from the sales team, collaborating with other engineers to improve the product, treating colleagues in the sales team with respect and professionalism, and contributing to the overall success of the company through effective communication and collaboration. Additionally, the product design team is responsible for optimizing the product portfolio, launching new products in high-demand markets, and enhancing post-sales support and customer service.

------------------------------------------------------------


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?